# 01 — Tokenization: from characters to Byte Pair Encoding

A language model never sees text. It sees **sequences of integers**, and the tokenizer is the
contract that converts between the two. Get this wrong and everything downstream suffers, so we
build it first — and we build it twice:

1. **Character-level** — 10 lines of code, gets us end-to-end *today*. We'll train our
   Shakespeare models with this.
2. **Byte-level BPE from scratch** — the algorithm behind GPT-2/3/4's tokenizers. We implement
   the textbook version, watch it be slow and slightly wrong, then fix both problems the way
   real tokenizers do. We'll use this for the TinyStories run (notebook 08).

**Why start character-level at all?** Because the fastest way to a *working* system is the
smallest one. A char tokenizer has no failure modes: no unknown tokens, no training step, a
vocab you can print on one screen. It lets us debug the *model* without wondering if the
*tokenizer* is the bug. Only once the pipeline works do we earn the right to make it fancier.

In [1]:
import sys, time
from pathlib import Path

ROOT = Path("..").resolve()          # repo root (notebooks/ -> repo)
sys.path.insert(0, str(ROOT))        # so we can import bpe.py later

text = (ROOT / "data" / "tinyshakespeare.txt").read_text()
print(f"corpus: {len(text):,} characters, {len(set(text))} unique")
print("-" * 60)
print(text[:300])

corpus: 1,115,394 characters, 65 unique
------------------------------------------------------------
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us


## The design space

Any tokenizer picks a point on one axis: **how much text does one integer represent?**

| granularity | vocab size | seq length for 1 page | fatal flaw |
|---|---|---|---|
| **characters/bytes** | ~65 / 256 | ~3,000 tokens | sequences are hugely long; model wastes capacity re-learning spelling |
| **subwords (BPE)** | 1k–100k | ~800 tokens | none fatal — this is why everyone uses it |
| **words** | 100k–1M+ | ~600 tokens | out-of-vocabulary words ("Grickle-grass") are unrepresentable; morphology invisible (`run`/`running` unrelated ids) |

Why sequence length matters so much: attention is **O(T²)** in time and memory (notebook 02),
and a model with a 256-token context sees **4× less text** if each token is a character rather
than a ~4-character subword. Subwords buy us reach.

Why not words: the vocab explodes, every typo is `<unk>`, and agglutinative languages
(Turkish, Finnish) have effectively infinite word forms. BPE sidesteps all of this by *learning*
its units from data: common words become single tokens, rare words decompose into pieces.

## Character-level tokenizer

The entire thing: sort the unique characters, number them. `encode` and `decode` are dictionary
lookups. Note we build the vocab **from the corpus** — this tokenizer can only represent
characters it has seen (fine here, since model training and generation both live on this corpus).

In [2]:
class CharTokenizer:
    def __init__(self, text):
        chars = sorted(set(text))                       # deterministic ordering
        self.stoi = {ch: i for i, ch in enumerate(chars)}  # string -> int
        self.itos = {i: ch for i, ch in enumerate(chars)}  # int -> string
    def encode(self, s):  return [self.stoi[c] for c in s]
    def decode(self, ids): return "".join(self.itos[i] for i in ids)
    @property
    def vocab_size(self): return len(self.stoi)

char_tok = CharTokenizer(text)
print("vocab size:", char_tok.vocab_size)
print("vocab:", repr("".join(char_tok.itos[i] for i in range(char_tok.vocab_size))))

ids = char_tok.encode("To be, or not to be")
print("\n'To be, or not to be' ->", ids)
assert char_tok.decode(ids) == "To be, or not to be"   # round-trip must be exact
print("round-trip OK")

vocab size: 65
vocab: "\n !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"

'To be, or not to be' -> [32, 53, 1, 40, 43, 6, 1, 53, 56, 1, 52, 53, 58, 1, 58, 53, 1, 40, 43]
round-trip OK


That's it. 65 symbols cover all of Shakespeare. Keep this class in mind as the baseline: **1
character per token**, so a 256-token context window holds 256 characters — about two sentences.

## Byte Pair Encoding: the idea

BPE builds a vocabulary bottom-up from data:

> Start with a base alphabet. Repeatedly find the **most frequent adjacent pair** of tokens in
> the corpus and **merge it into a new token**. Stop when the vocab is as big as you want.

After a few hundred merges, frequent character runs like `the`, `ing`, ` and` have become single
tokens, while rare words still decompose into pieces (ultimately into single bytes). Frequency
decides everything — the vocab automatically adapts to whatever corpus you train on.

**Why bytes as the base alphabet, not characters?** If the base units are the 256 possible byte
values, then *any* string — emoji, Chinese, typos, binary garbage — encodes as a valid sequence.
There is **no `<unk>` token, ever**. Unicode has ~150k codepoints, so using characters as the
base would either blow up the vocab or reintroduce unknowns. This byte-level trick is exactly
what GPT-2 introduced, and we copy it.

## The textbook algorithm (naive version)

Two helpers do all the work:

- `get_stats(ids)` — count every adjacent pair.
- `merge(ids, pair, new_id)` — replace each occurrence of `pair` with a fresh id.

Training = loop: count, pick the max, merge, repeat. New ids start at 256 (0–255 are the raw bytes).

In [3]:
def get_stats(ids, counts=None):
    '''Count adjacent pairs: [1,2,3,1,2] -> {(1,2):2, (2,3):1, (3,1):1}'''
    counts = {} if counts is None else counts
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, new_id):
    '''Replace every occurrence of `pair` in `ids` with `new_id`.'''
    out, i = [], 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            out.append(new_id); i += 2
        else:
            out.append(ids[i]); i += 1
    return out

# micro-example: watch one merge happen
demo = list("aababa".encode("utf-8"))
print("bytes:  ", demo)
print("stats:  ", get_stats(demo))
print("merged: ", merge(demo, (97, 98), 256), "   # every 'ab' became token 256")

bytes:   [97, 97, 98, 97, 98, 97]
stats:   {(97, 97): 1, (97, 98): 2, (98, 97): 2}
merged:  [97, 256, 256, 97]    # every 'ab' became token 256


In [4]:
def naive_bpe_train(text, num_merges, verbose=False):
    '''Textbook BPE on the raw byte stream.'''
    ids = list(text.encode("utf-8"))
    merges = {}
    for i in range(num_merges):
        stats = get_stats(ids)
        # deterministic tie-break: highest count, then lowest byte ids
        pair = max(stats, key=lambda p: (stats[p], (-p[0], -p[1])))
        new_id = 256 + i
        ids = merge(ids, pair, new_id)
        merges[pair] = new_id
        if verbose:
            print(f"merge {i+1:2d}: {pair} -> {new_id}  (count {stats[pair]:,})")
    return merges

t0 = time.time()
naive_merges = naive_bpe_train(text[:50_000], num_merges=30, verbose=True)
naive_time = time.time() - t0
print(f"\n30 merges on 50KB took {naive_time:.1f}s")

merge  1: (101, 32) -> 256  (count 1,347)
merge  2: (116, 104) -> 257  (count 1,064)
merge  3: (115, 32) -> 258  (count 734)
merge  4: (116, 32) -> 259  (count 716)
merge  5: (111, 117) -> 260  (count 645)
merge  6: (44, 32) -> 261  (count 576)
merge  7: (100, 32) -> 262  (count 572)
merge  8: (101, 114) -> 263  (count 537)
merge  9: (105, 110) -> 264  (count 449)
merge 10: (97, 110) -> 265  (count 446)
merge 11: (32, 257) -> 266  (count 420)
merge 12: (101, 110) -> 267  (count 379)
merge 13: (58, 10) -> 268  (count 378)
merge 14: (111, 114) -> 269  (count 378)
merge 15: (97, 114) -> 270  (count 368)
merge 16: (10, 10) -> 271  (count 339)
merge 17: (121, 32) -> 272  (count 339)
merge 18: (111, 110) -> 273  (count 330)
merge 19: (108, 108) -> 274  (count 298)
merge 20: (111, 32) -> 275  (count 275)
merge 21: (104, 97) -> 276  (count 264)
merge 22: (121, 260) -> 277  (count 264)
merge 23: (101, 115) -> 278  (count 243)
merge 24: (46, 271) -> 279  (count 237)
merge 25: (44, 10) -> 280  (c

merge 27: (104, 105) -> 282  (count 214)
merge 28: (85, 83) -> 283  (count 202)
merge 29: (266, 256) -> 284  (count 200)
merge 30: (283, 268) -> 285  (count 199)

30 merges on 50KB took 0.2s


It works — you can watch it discover `e␣`, `th`, `t␣`... But look closely at what it learned and
how long it took. Two real problems:

**Problem 1 — it merges across word boundaries.** Pairs like `e` + `␣` (letter followed by
space) or even `␣the␣` glued to the next word's first letter are frequent, so the naive
algorithm happily spends vocabulary slots on them. These tokens encode *accidents of adjacency*,
not reusable units of meaning: the token for `t␣he` helps nothing else. GPT-2's fix is to
**pre-split the text with a regex** into "chunks" — words, numbers, punctuation runs, whitespace —
and forbid merges from crossing chunk boundaries. One important subtlety: the space is attached
to the *front* of the following word, so ` the` (with leading space) is one chunk. That way the
vocabulary naturally contains ` the`, ` and`, ... and no merges are wasted gluing spaces on.

**Problem 2 — it's absurdly slow.** Every merge re-scans the entire corpus: `O(corpus)` work per
merge, so training 4,000 merges on 1GB is hopeless. The fix exploits a simple fact: once you've
pre-split into chunks, *identical chunks tokenize identically*. Shakespeare has 1.1M characters
but only ~14k **distinct** chunks. So: count each distinct chunk once with its frequency, run
merges on that tiny weighted set, and — the second trick — update pair counts **incrementally**,
touching only the chunks that actually contain the merged pair (we keep an index `pair -> chunks
containing it`). This is how Sennrich et al.'s original BPE implementation works.

In [5]:
import regex   # the `regex` module supports \p{L} (any-language letter); stdlib `re` doesn't

# GPT-2's pre-tokenization pattern, piece by piece:
#   '(?:[sdmt]|ll|ve|re)   English contractions ('s, 'll, 've, ...) split off as units
#   ?\p{L}+                a run of letters, with an OPTIONAL LEADING SPACE
#   ?\p{N}+                a run of digits
#   ?[^\s\p{L}\p{N}]+      a run of punctuation/symbols
#   \s+(?!\S) | \s+        whitespace runs
GPT2_SPLIT_PATTERN = r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

print(regex.findall(GPT2_SPLIT_PATTERN, "Thou art123 a knave, and I'll prove it!"))

['Thou', ' art', '123', ' a', ' knave', ',', ' and', ' I', "'ll", ' prove', ' it', '!']


Note ` art`, ` a`, ` knave` carry their leading space, `'ll` split off on its own, digits and
punctuation separated. Merges will now happen *within* these chunks only.

## The fast trainer

Below is the full implementation (this identical class lives in [`bpe.py`](../bpe.py) so later
notebooks can import it). Read the `train` method against the plan above:

1. regex-split → `Counter` of distinct chunks with frequencies,
2. initial pair counts + an inverted index `pair_to_seqs`,
3. per merge: pick the max pair, then rewrite **only the chunks containing it**, subtracting
   their old pair counts and adding the new ones.

`encode` has its own subtlety: at encode time we must replay merges **in the order they were
learned** (lowest merge-id first). If we applied them in any other order we'd get different
tokens than training did, and the learned statistics would be off.

In [6]:
import json
from collections import Counter, defaultdict

class BPETokenizer:
    '''Byte-level BPE with GPT-2-style regex pre-splitting and an incremental trainer.'''

    def __init__(self):
        self.merges = {}                                   # (id, id) -> merged id
        self.vocab = {i: bytes([i]) for i in range(256)}   # id -> raw bytes
        self.pattern = GPT2_SPLIT_PATTERN

    # ---------------------------------------------------------- training
    def train(self, text, vocab_size, verbose=False):
        assert vocab_size >= 256
        num_merges = vocab_size - 256

        # 1) distinct chunks, each processed ONCE, weighted by frequency
        chunk_freqs = Counter(regex.findall(self.pattern, text))
        seqs  = [list(chunk.encode("utf-8")) for chunk in chunk_freqs]
        freqs = list(chunk_freqs.values())

        # 2) initial pair counts + inverted index: pair -> chunk indices containing it
        pair_counts, pair_to_seqs = defaultdict(int), defaultdict(set)
        for si, (seq, f) in enumerate(zip(seqs, freqs)):
            for pair in zip(seq, seq[1:]):
                pair_counts[pair] += f
                pair_to_seqs[pair].add(si)

        for i in range(num_merges):
            if not pair_counts:
                print(f"ran out of pairs after {i} merges"); break
            pair = max(pair_counts, key=lambda p: (pair_counts[p], (-p[0], -p[1])))
            new_id = 256 + i
            self.merges[pair] = new_id
            self.vocab[new_id] = self.vocab[pair[0]] + self.vocab[pair[1]]

            # 3) incremental update: rewrite only affected chunks
            for si in list(pair_to_seqs[pair]):
                seq, f = seqs[si], freqs[si]
                for p in zip(seq, seq[1:]):            # retract old pair counts
                    pair_counts[p] -= f
                    if pair_counts[p] <= 0: del pair_counts[p]
                    pair_to_seqs[p].discard(si)
                seq = merge(seq, pair, new_id)
                seqs[si] = seq
                for p in zip(seq, seq[1:]):            # add new pair counts
                    pair_counts[p] += f
                    pair_to_seqs[p].add(si)

            if verbose and (i + 1) % 64 == 0:
                print(f"  merge {i+1}/{num_merges}: {self.vocab[new_id]!r}")

    # ---------------------------------------------------------- encode/decode
    def _encode_chunk(self, chunk_bytes):
        ids = list(chunk_bytes)
        while len(ids) >= 2:
            # apply the EARLIEST-LEARNED applicable merge first (training order!)
            stats = get_stats(ids)
            pair = min(stats, key=lambda p: self.merges.get(p, float("inf")))
            if pair not in self.merges: break
            ids = merge(ids, pair, self.merges[pair])
        return ids

    def encode(self, text):
        ids = []
        for chunk in regex.findall(self.pattern, text):
            ids.extend(self._encode_chunk(chunk.encode("utf-8")))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids)
        # errors='replace': sampling can split a multi-byte utf-8 char; show
        # U+FFFD rather than crash
        return data.decode("utf-8", errors="replace")

    @property
    def vocab_size(self): return 256 + len(self.merges)

    def save(self, path):
        obj = {"pattern": self.pattern,
               "merges": [[a, b, v] for (a, b), v in self.merges.items()]}
        Path(path).write_text(json.dumps(obj))

    @classmethod
    def load(cls, path):
        obj = json.loads(Path(path).read_text())
        tok = cls(); tok.pattern = obj["pattern"]
        for a, b, v in obj["merges"]:      # replay in order to rebuild vocab
            tok.merges[(a, b)] = v
            tok.vocab[v] = tok.vocab[a] + tok.vocab[b]
        return tok

In [7]:
tok = BPETokenizer()
t0 = time.time()
tok.train(text, vocab_size=512, verbose=True)      # FULL 1.1MB corpus, 256 merges
fast_time = time.time() - t0

naive_rate = naive_time / (30 * 50_000)            # seconds per (merge x char)
naive_extrapolated = naive_rate * 256 * len(text)  # what the naive version would need
print(f"\nfast trainer: 256 merges on {len(text)/1e6:.1f}MB in {fast_time:.2f}s")
print(f"naive trainer, extrapolated to the same job: ~{naive_extrapolated/60:.0f} minutes")

  merge 64/256: b'ith'
  merge 128/256: b' do'
  merge 192/256: b' good'
  merge 256/256: b'ather'

fast trainer: 256 merges on 1.1MB in 0.34s
naive trainer, extrapolated to the same job: ~1 minutes


In [8]:
# What did it learn? Longest tokens in the vocab -- recognizable English units,
# each with its leading space, and NO cross-word garbage:
longest = sorted(tok.vocab.values(), key=len, reverse=True)[:20]
print([t.decode("utf-8", errors="replace") for t in longest])

[' shall', ' there', ' would', ' that', ' with', ' your', ' thou', ' have', ' this', ' will', ' thee', ' what', ' good', ' lord', ' from', ' more', ' them', 'other', ' know', 'ather']


In [9]:
# See an actual tokenization (| marks token boundaries):
sample = "Friends, Romans, countrymen, lend me your ears;"
pieces = [tok.vocab[i].decode("utf-8", errors="replace") for i in tok.encode(sample)]
print("|".join(pieces))
print(f"\n{len(sample)} chars -> {len(pieces)} BPE tokens  (vs {len(sample)} char-level tokens)")

F|ri|end|s|,| R|om|an|s|,| c|ou|nt|ry|m|en|,| l|end| me| your| e|ar|s|;

47 chars -> 25 BPE tokens  (vs 47 char-level tokens)


In [10]:
# Round-trip must be EXACT, including text the tokenizer never saw in training.
# This is the byte-level payoff: emoji and accents fall back to raw bytes, no <unk>.
tests = [
    "To be, or not to be: that is the question.",
    "naive, cafe, uber — 🙂🎭 中文 (none of this is in Shakespeare)",
    "  weird   spacing\n\nand\tnewlines",
]
for s in tests:
    assert tok.decode(tok.encode(s)) == s, f"round-trip failed: {s!r}"
print("all round-trips exact")

enc = tok.encode("🙂")
print("'🙂' ->", enc, "-> raw bytes, merged nothing, decoded fine:", tok.decode(enc))

all round-trips exact
'🙂' -> [240, 159, 153, 130] -> raw bytes, merged nothing, decoded fine: 🙂


## How good is the compression?

The number that matters is **characters per token** — it directly multiplies how much text fits
in the model's context window. Compare our 512-token vocab against char-level and against
GPT-2's real tokenizer (50,257 tokens, trained on web text via `tiktoken`):

In [11]:
import tiktoken
gpt2 = tiktoken.get_encoding("gpt2")

probe = text[:100_000]
rows = [
    ("char-level (65)",        len(probe)),
    ("our BPE (512)",          len(tok.encode(probe))),
    ("GPT-2 BPE (50,257)",     len(gpt2.encode(probe))),
]
print(f"{'tokenizer':24s} {'tokens':>10s} {'chars/token':>12s}")
for name, n in rows:
    print(f"{name:24s} {n:>10,d} {len(probe)/n:>12.2f}")

tokenizer                    tokens  chars/token
char-level (65)             100,000         1.00
our BPE (512)                51,511         1.94
GPT-2 BPE (50,257)           30,645         3.26


Two things worth noticing:

- Even a tiny 512-entry vocab roughly **halves** sequence length vs characters. GPT-2's 50k
  vocab gets ~4 chars/token — diminishing returns kick in fast, which is why vocabs stopped
  growing far beyond ~100k.
- GPT-2's tokenizer wins despite being trained on *web text*, not Shakespeare — 100× more vocab
  slots buys a lot. Per vocabulary slot, our corpus-specific tokenizer is far more efficient.
  This is the general trade-off: **bigger vocab ⇒ shorter sequences but a bigger embedding
  matrix and rarer, less-trained tokens.**

## What we use downstream — and why

| run | tokenizer | why |
|---|---|---|
| TinyShakespeare (notebooks 04–07) | **char-level, 65** | the corpus is tiny; a 65-row embedding keeps the model small, and losses are directly comparable across all our experiments. With ~0.8M params we'd rather spend capacity on the transformer than on a vocab. |
| TinyStories (notebook 08) | **our BPE, 4,096** | 2GB corpus, a 25M-param model, and stories long enough that the ~4× context reach matters. The model stops wasting layers on spelling and spends them on grammar and narrative. |

We save the Shakespeare-trained tokenizer as an artifact:

In [12]:
out = ROOT / "data" / "shakespeare_bpe_512.json"
tok.save(out)
reloaded = BPETokenizer.load(out)
assert reloaded.encode("thy kingdom come") == tok.encode("thy kingdom come")
print("saved and reloaded:", out.name)

saved and reloaded: shakespeare_bpe_512.json


## Takeaways

- A tokenizer is a **lossless** text↔ints codec; round-trip exactness is the invariant to test.
- **BPE = greedy frequency-based merging.** The vocab is *learned from a corpus*, so it adapts
  to the domain — and inherits its biases (a tokenizer trained on English fragments other
  languages into many tokens).
- **Byte-level base alphabet ⇒ no `<unk>`, ever.** Anything encodes; rare things just cost more tokens.
- The regex pre-split keeps merges inside word-like chunks — vocabulary slots go to reusable
  units, and the leading-space convention means no merges are wasted attaching spaces.
- The fast trainer is the same algorithm computed cleverly: **dedup chunks + incremental counts**
  turned minutes into a fraction of a second. (Real tokenizers go further and do this in Rust —
  our pure-Python `encode` is the reason notebook 08 caches encoded chunks.)

**Interview quick-fire:**
- *Why does GPT-2 tokenize at the byte level?* → 256-symbol closed alphabet: any string encodes, no unknown tokens, vocab stays bounded.
- *Why is `" the"` (with space) a single token?* → the regex attaches the space to the following word, so word tokens absorb their separator and sequences shorten.
- *Why can't you use someone else's model with your own tokenizer?* → token ids index the embedding matrix; change the mapping and every id points at the wrong learned vector.

**Next:** [02 — Attention](02_attention.ipynb), where these token ids start talking to each other.